# Data Lake Assignment
**Name:** Akshay Kumar  
**Roll Number:** LCB2023007

Keep `sales.csv` unchanged in Bronze, check its quality using Spark, and save a cleaned copy in Silver.
Run the cells in order in Google Colab and upload the supplied `sales.csv` when prompted. For a local run, start from the repository root with the dataset in `dataset/sales.csv`.

## Setup

In [1]:
%pip install -q pyspark==3.5.8

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]")
         .appName("SalesPreprocessing")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

## Bronze copy
The source file is copied without editing its contents. The Silver directory holds Spark's CSV output.

In [3]:
bronze_dir = Path("data_lake/bronze")
silver_dir = Path("data_lake/silver/preprocessed_sales")
bronze_dir.mkdir(parents=True, exist_ok=True)
silver_dir.mkdir(parents=True, exist_ok=True)

source_file = Path("dataset/sales.csv")
if not source_file.exists():
    source_file = Path("sales.csv")
if not source_file.exists():
    from google.colab import files
    uploaded = files.upload()
    if "sales.csv" not in uploaded:
        raise ValueError("Upload the original file named sales.csv.")

bronze_file = bronze_dir / "sales.csv"
shutil.copyfile(source_file, bronze_file)

Out[0]: PosixPath('data_lake/bronze/sales.csv')


## Load and inspect

In [4]:
# Read as strings first so the original values can be checked before casting.
bronze = spark.read.option("header", True).csv(str(bronze_file))
bronze.printSchema()
bronze.show(5, truncate=False)
original_count = bronze.count()
print("Bronze rows:", original_count)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+---

## Quality checks

In [5]:
distinct_count = bronze.dropDuplicates().count()
duplicate_count = original_count - distinct_count
print("Exact duplicate rows:", duplicate_count)
bronze.groupBy(bronze.columns).count().filter(F.col("count") > 1).show(5, truncate=False)

Exact duplicate rows: 10
+--------+-----------+--------------+---------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name |product        |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+--------------+---------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100444  |C0090      |Abhinav Pillai|Novel          |Books    |8       |530       |15              |Credit Card     |Mumbai    |Maharashtra   |2025-09-11|Delivered   |2    |
|100773  |C0315      |Meera Malhotra|Study Table    |Furniture|7       |11310     |40              |Debit Card      |Pune      |Maharashtra   |2026-02-25|Delivered   |2    |
|100297  |C0149      |Manish Pillai |Dry Fruits Pack|Grocery  |7       |780       |35              |Credi

In [6]:
checks = []
for name in bronze.columns:
    missing = F.col(name).isNull() | (F.trim(F.col(name)) == "")
    extra_space = F.col(name) != F.trim(F.col(name))
    checks.extend([
        F.sum(F.when(missing, 1).otherwise(0)).alias(name + "_missing"),
        F.sum(F.when(extra_space, 1).otherwise(0)).alias(name + "_spaces")
    ])

counts = bronze.agg(*checks).first().asDict()
spark.createDataFrame([
    (name, counts[name + "_missing"], counts[name + "_spaces"])
    for name in bronze.columns
], ["column", "null_or_blank", "extra_spaces"]).show(len(bronze.columns), truncate=False)

for name in ["category", "city", "payment_method"]:
    print(name)
    bronze.groupBy(name).count().orderBy(name).show(30, truncate=False)

+----------------+-------------+------------+
|column          |null_or_blank|extra_spaces|
+----------------+-------------+------------+
|order_id        |0            |0           |
|customer_id     |0            |0           |
|customer_name   |10           |3           |
|product         |0            |0           |
|category        |0            |0           |
|quantity        |0            |0           |
|unit_price      |0            |0           |
|discount_percent|0            |0           |
|payment_method  |8            |0           |
|city            |8            |3           |
|state           |0            |0           |
|order_date      |0            |0           |
|order_status    |0            |0           |
+----------------+-------------+------------+

category
+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |136  |
|Books          |128  |
|CLOTHING       |1    |
|Clothing       |108  |
|ELECTRONICS    |1    |
|Electronics    

In [7]:
quantity = F.col("quantity").cast("double")
price = F.col("unit_price").cast("double")
discount = F.col("discount_percent").cast("double")
valid_quantity = (quantity > 0) & (quantity == F.floor(quantity))
valid_price = price > 0
valid_discount = discount.between(0, 100)

numeric_rules = {
    "quantity": valid_quantity,
    "unit_price": valid_price,
    "discount_percent": valid_discount
}
for name, rule in numeric_rules.items():
    bad_rows = bronze.filter(~F.coalesce(rule, F.lit(False)))
    print(name, "missing or invalid:", bad_rows.count())
    bad_rows.groupBy(name).count().orderBy(name).show(truncate=False)

quantity missing or invalid: 12
+--------+-----+
|quantity|count|
+--------+-----+
|-1      |1    |
|-2      |5    |
|0       |6    |
+--------+-----+

unit_price missing or invalid: 8
+----------+-----+
|unit_price|count|
+----------+-----+
|-250      |1    |
|-500      |3    |
|0         |4    |
+----------+-----+

discount_percent missing or invalid: 8
+----------------+-----+
|discount_percent|count|
+----------------+-----+
|-5              |4    |
|120             |4    |
+----------------+-----+



In [8]:
date_text = F.trim(F.col("order_date"))
date_format = (F.when(date_text.rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
               .when(date_text.rlike(r"^\d{4}/\d{2}/\d{2}$"), "yyyy/MM/dd")
               .when(date_text.rlike(r"^\d{2}-\d{2}-\d{4}$"), "MM-dd-yyyy")
               .when(date_text.rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
               .otherwise("unrecognized"))
bronze.groupBy(date_format.alias("date_format")).count().show(truncate=False)
bronze.filter(~date_text.rlike(r"^\d{4}-\d{2}-\d{2}$")).select("order_id", "order_date").show()

parsed_date = F.coalesce(
    F.to_date(date_text, "yyyy-MM-dd"),
    F.to_date(date_text, "yyyy/MM/dd"),
    F.to_date(date_text, "MM-dd-yyyy"),
    F.to_date(date_text, "dd/MM/yyyy")
)
date_check = bronze.withColumn("parsed_date", parsed_date)
print("Unparseable dates:", date_check.filter(F.col("parsed_date").isNull()).count())
date_check.filter(F.col("parsed_date").isNull()).select("order_id", "order_date").show()

+------------+-----+
|date_format |count|
+------------+-----+
|yyyy-MM-dd  |996  |
|yyyy/MM/dd  |1    |
|MM-dd-yyyy  |1    |
|dd/MM/yyyy  |1    |
|unrecognized|1    |
+------------+-----+

+--------+----------+
|order_id|order_date|
+--------+----------+
|  100097|31/02/2026|
|  100358|2025/07/14|
|  100448|not-a-date|
|  100969|09-15-2026|
+--------+----------+

Unparseable dates: 4
+--------+----------+
|order_id|order_date|
+--------+----------+
|  100097|31/02/2026|
|  100448|not-a-date|
|  100634|2025-02-30|
|  100637|2026-13-05|
+--------+----------+



## Preprocessing
Only category and city need capitalization changes; product names and `UPI` keep their original spelling. Missing names, cities, and payment methods become `Unknown`.

Quantity must be a positive integer, price must be positive, and discount must be between 0 and 100. Invalid numbers and dates become null, keeping the rest of each record without guessing replacement values. Positive prices are kept even when they are unusually high.

In [9]:
silver = bronze.dropDuplicates()
for name in bronze.columns:
    silver = silver.withColumn(name, F.trim(F.col(name)))

for name in ["category", "city"]:
    silver = silver.withColumn(name, F.initcap(F.lower(F.col(name))))

for name in ["customer_name", "city", "payment_method"]:
    silver = silver.withColumn(
        name, F.when(F.col(name).isNull() | (F.col(name) == ""), "Unknown")
               .otherwise(F.col(name))
    )

silver = (silver
    .withColumn("quantity", F.when(valid_quantity, F.col("quantity").cast("int")))
    .withColumn("unit_price", F.when(valid_price, F.col("unit_price").cast("decimal(12,2)")))
    .withColumn("discount_percent", F.when(valid_discount, F.col("discount_percent").cast("decimal(5,2)")))
    .withColumn("order_date", parsed_date))

## Save Silver

In [10]:
# Sorting makes the small output file easy to inspect.
silver.orderBy("order_id", "customer_id").coalesce(1).write.mode("overwrite").option("header", True).csv(str(silver_dir))

roll_number = "LCB2023007"
student_name = "AkshayKumar"
submission_dir = Path("submissions") / f"{roll_number}_{student_name}"
submission_silver = submission_dir / "silver"
submission_silver.mkdir(parents=True, exist_ok=True)
part_file = next(silver_dir.glob("part-*.csv"))
output_file = submission_silver / "sales_silver.csv"
shutil.copyfile(part_file, output_file)
print("Saved:", output_file)

Saved: submissions/LCB2023007_AkshayKumar/silver/sales_silver.csv


## Verify

In [11]:
# Keep the numeric and date types when reading the CSV back.
saved_silver = spark.read.option("header", True).schema(silver.schema).csv(str(silver_dir))
final_count = saved_silver.count()
print("Bronze rows:", original_count)
print("Silver rows:", final_count)
print("Duplicate rows removed:", duplicate_count)
saved_silver.orderBy("order_id", "customer_id").show(5, truncate=False)
saved_silver.printSchema()

saved_silver.select([
    F.sum(F.when(F.col(name).isNull(), 1).otherwise(0)).alias(name)
    for name in ["quantity", "unit_price", "discount_percent", "order_date"]
]).show()

print("Silver data saved and read back using Spark.")

Bronze rows: 1000
Silver rows: 990
Duplicate rows removed: 10
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710.00   |20.00           |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680.00    |5.00            |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080.00   |12.00           |UPI           |Pune      |

## Summary
Removed exact duplicates, trimmed spaces, corrected category and city capitalization, and filled missing descriptive fields with `Unknown`. Invalid numbers and dates were left null. Valid dates use `yyyy-MM-dd`, and the cleaned data was saved and read back from Silver using Spark.